# Kinyarwanda Sentiment Classification: Ablations

Two cheap ablations on AfroXLMR-base (learning rate 2e-5, three seeds each), to justify two design choices instead of just asserting them:

- **Class weights**: weighted cross-entropy (used everywhere) against plain cross-entropy.
- **Casing**: the Transformer notebooks keep case, while the TF-IDF and BiLSTM notebooks lowercase. Cased input (reference) against lowercased input.

The reference configuration (cased, weighted) is read from `finetune_runs.json` if notebook 03 has been run, otherwise it is trained here. About an hour on a T4.

## Setup

In [1]:
!pip install -q transformers

In [2]:
import re, os, json, random, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu")

Device: Tesla T4


## Data

In [3]:
base = "hf://datasets/shmuhammad/AfriSenti-twitter-sentiment@refs%2Fconvert%2Fparquet/kin"
ds = load_dataset("parquet", data_files={
    "train":      f"{base}/train/0000.parquet",
    "validation": f"{base}/validation/0000.parquet",
    "test":       f"{base}/test/0000.parquet",
})
label_names = ds["train"].features["label"].names

def clean(text, lower=False):
    t = re.sub(r"\s+", " ", re.sub(r"@user", " ", text)).strip()
    return t.lower() if lower else t

train_df, val_df, test_df = (ds[s].to_pandas() for s in ("train", "validation", "test"))
y_train, y_val, y_test = (d["label"].tolist() for d in (train_df, val_df, test_df))
print(len(train_df), len(val_df), len(test_df))

kin/train/0000.parquet: reconstructing file:   0%|          |  0.00B /  270kB            

kin/train/0000.parquet: downloading bytes:           |  0.00B            

kin/validation/0000.parquet: reconstructing file:   0%|          |  0.00B / 66.8kB            

kin/validation/0000.parquet: downloading bytes:           |  0.00B            

kin/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 86.3kB            

kin/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

3302 827 1026


## Training code
The same loop as in notebook 03, with two switches: `lower` for the casing of the input and `weighted` for the class weights.

In [4]:
MAX_LEN = 128
BATCH_SIZE, EVAL_BATCH, EPOCHS = 16, 64, 5
WARMUP, WEIGHT_DECAY = 0.1, 0.01
use_amp = device.type == "cuda"

counts = np.bincount(y_train)
class_weights = torch.tensor(len(y_train) / (len(label_names) * counts), dtype=torch.float32).to(device)

_cache = {}
def get_data(hub_id, lower):
    if (hub_id, lower) not in _cache:
        t = AutoTokenizer.from_pretrained(hub_id)
        enc = lambda df: t([clean(x, lower) for x in df["tweet"]], truncation=True, max_length=MAX_LEN)["input_ids"]
        _cache[(hub_id, lower)] = (t.pad_token_id, enc(train_df), enc(val_df), enc(test_df))
    return _cache[(hub_id, lower)]

In [5]:
def make_batches(ids, labels, batch_size, shuffle, pad_id):
    order = np.random.permutation(len(ids)) if shuffle else np.arange(len(ids))
    for s in range(0, len(ids), batch_size):
        idx = order[s:s + batch_size]
        seqs = [ids[i] for i in idx]
        width = max(len(q) for q in seqs)
        x = torch.full((len(seqs), width), pad_id, dtype=torch.long)
        mask = torch.zeros((len(seqs), width), dtype=torch.long)
        for j, q in enumerate(seqs):
            x[j, :len(q)] = torch.tensor(q)
            mask[j, :len(q)] = 1
        y = torch.tensor([labels[i] for i in idx])
        yield x.to(device), mask.to(device), y.to(device)

def predict(model, ids, labels, pad_id):
    model.eval()
    preds = []
    with torch.no_grad(), torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
        for x, mask, _ in make_batches(ids, labels, EVAL_BATCH, False, pad_id):
            preds += model(input_ids=x, attention_mask=mask).logits.argmax(-1).tolist()
    return preds

def scores(y_true, y_pred):
    _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_f1": f1}

In [6]:
def run(hub_id, lr, seed, lower=False, weighted=True):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    pad_id, tr, va, te = get_data(hub_id, lower)
    model = AutoModelForSequenceClassification.from_pretrained(
        hub_id, num_labels=len(label_names),
        id2label=dict(enumerate(label_names)),
        label2id={n: i for i, n in enumerate(label_names)}).float().to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * ((len(tr) + BATCH_SIZE - 1) // BATCH_SIZE)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights if weighted else None)

    best, history = {"val_macro_f1": -1}, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for x, mask, y in make_batches(tr, y_train, BATCH_SIZE, True, pad_id):
            optimizer.zero_grad()
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=x, attention_mask=mask).logits
            scaler.scale(loss_fn(logits.float(), y)).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step()

        val_f1 = scores(y_val, predict(model, va, y_val, pad_id))["macro_f1"]
        history.append(val_f1)
        if val_f1 > best["val_macro_f1"]:
            test = scores(y_test, predict(model, te, y_test, pad_id))
            best = {"val_macro_f1": val_f1, "best_epoch": epoch,
                    "test_macro_f1": test["macro_f1"], "test_accuracy": test["accuracy"]}
    best["history"] = history
    del model, optimizer
    gc.collect(); torch.cuda.empty_cache()
    return best

## Experiments

In [7]:
HUB, LR = "Davlan/afro-xlmr-base", 2e-5
SEEDS = [42, 43, 44]
VARIANTS = [
    ("Reference: cased, class weights", dict(lower=False, weighted=True)),
    ("No class weights",                dict(lower=False, weighted=False)),
    ("Lowercased input",                dict(lower=True,  weighted=True)),
]

ablation = json.load(open("ablation_runs.json")) if os.path.exists("ablation_runs.json") else {}
for name, kw in VARIANTS:
    if name in ablation:
        continue
    if name.startswith("Reference") and os.path.exists("finetune_runs.json"):
        ref = json.load(open("finetune_runs.json")).get("AfroXLMR-base, lr 2e-5")
        if ref:
            ablation[name] = [{k: v for k, v in x.items() if k != "test_preds"} for x in ref]
            continue
    t0 = time.time()
    ablation[name] = [run(HUB, LR, s, **kw) for s in SEEDS]
    json.dump(ablation, open("ablation_runs.json", "w"))
    r = ablation[name]
    print(f"{name}: val F1 {np.mean([x['val_macro_f1'] for x in r]):.3f} | test F1 {np.mean([x['test_macro_f1'] for x in r]):.3f} | {(time.time() - t0) / 60:.1f} min")
json.dump(ablation, open("ablation_runs.json", "w"))

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Lowercased input: val F1 0.669 | test F1 0.668 | 9.4 min


## Optional: is XLM-R base undertrained?
In notebook 03 XLM-R base picked its last epoch (5) in all three seeds, and its validation score was still rising. This cell trains it for 10 epochs (same learning rate, schedule stretched to 10 epochs) to see whether the gap to AfroXLMR-base shrinks. About 20 minutes on a T4. Skip it if time is short and state the limitation in the report.

In [ ]:
EPOCHS_5 = EPOCHS
EPOCHS = 10
name = "XLM-R base, 10 epochs"
if name not in ablation:
    ablation[name] = [run("FacebookAI/xlm-roberta-base", 2e-5, s) for s in SEEDS]
    json.dump(ablation, open("ablation_runs.json", "w"))
EPOCHS = EPOCHS_5
r = ablation[name]
print(f"{name}: val F1 {np.mean([x['val_macro_f1'] for x in r]):.3f} | test F1 {np.mean([x['test_macro_f1'] for x in r]):.3f} | best epochs {[x['best_epoch'] for x in r]}")
print("Reference from notebook 03 (5 epochs): test F1 0.540 +- 0.019; AfroXLMR-base 0.663 +- 0.008")

## Results

In [8]:
def mean_std(r, key):
    v = [x[key] for x in r]
    return f"{np.mean(v):.3f} ± {np.std(v):.3f}"   # population std over three seeds

table = pd.DataFrame([{"variant": n, "val macro-F1": mean_std(r, "val_macro_f1"),
                       "test macro-F1": mean_std(r, "test_macro_f1"), "test accuracy": mean_std(r, "test_accuracy"),
                       "per-seed test macro-F1": [round(x["test_macro_f1"], 3) for x in r]}
                      for n, r in ablation.items()]).set_index("variant")
table.to_csv("ablation_results.csv")
table

,val macro-F1,test macro-F1,test accuracy,per-seed test macro-F1
variant,,,,
"Reference: cased, class weights",0.669 ± 0.004,0.663 ± 0.008,0.661 ± 0.007,"[0.663, 0.672, 0.654]"
No class weights,0.674 ± 0.005,0.671 ± 0.004,0.668 ± 0.004,"[0.67, 0.667, 0.675]"
Lowercased input,0.669 ± 0.003,0.668 ± 0.012,0.665 ± 0.011,"[0.68, 0.672, 0.651]"


### Observations
*Draft written from the printed numbers. Rewrite it in your own words and check it against the output before using it.*

- Test macro-F1 is 0.663 ± 0.008 with class weights, 0.671 ± 0.004 without, and 0.668 ± 0.012 with lowercased input. The per-seed values overlap (0.654 to 0.680), so **neither choice changes the result by more than the seed-to-seed spread**.
- Class weights are not needed here: the classes are only mildly imbalanced (neutral 38%, negative 35%, positive 27% of the test set). They are kept because they were fixed before the experiments, not because they help.
- Lowercasing does not matter for AfroXLMR-base, so using lowercased text for the TF-IDF and BiLSTM models and cased text for the Transformers does not bias the comparison between them.
- Limits: base model only, three seeds, one learning rate.